## Bkg plots: using coffea-casa

In [1]:
#### python
import os
import sys
import importlib
# columnar analysis
from coffea import processor
from coffea.nanoevents import NanoAODSchema
import awkward as ak
from dask.distributed import Client, performance_report
# local
sidm_path = str(os.getcwd()).split("/sidm")[0]
if sidm_path not in sys.path: sys.path.insert(1, sidm_path)
from sidm.tools import utilities, sidm_processor, scaleout, llpnanoaodschema
# always reload local modules to pick up changes during development
importlib.reload(utilities)
importlib.reload(sidm_processor)
importlib.reload(scaleout)
# plotting
import matplotlib.pyplot as plt
utilities.set_plot_style()
%matplotlib inline
import coffea.util
from matplotlib.colors import LogNorm
from coffea.processor import accumulate

In [2]:
# client = scaleout.make_dask_client("tls://localhost:8786")
# client

In [3]:
vr = "30"
nfiles = 50

sig2mu = [
    # "2Mu2E_200GeV_5p0GeV_100p0mm", "2Mu2E_500GeV_5p0GeV_80p0mm", "2Mu2E_800GeV_5p0GeV_50p0mm", "2Mu2E_1000GeV_5p0GeV_40p0mm", 
    # "2Mu2E_500GeV_0p25GeV_4p0mm", # "2Mu2E_500GeV_1p2GeV_19p0mm", "2Mu2E_500GeV_5p0GeV_80p0mm",
    # "2Mu2E_500GeV_1p2GeV_0p019mm", "2Mu2E_500GeV_1p2GeV_0p19mm", "2Mu2E_500GeV_1p2GeV_1p9mm", "2Mu2E_500GeV_1p2GeV_9p6mm", 
    "2Mu2E_500GeV_1p2GeV_19p0mm", ]

sig4mu = [
    # "4Mu_200GeV_5p0GeV_200p0mm", "4Mu_500GeV_5p0GeV_80p0mm", "4Mu_800GeV_5p0GeV_50p0mm", "4Mu_1000GeV_5p0GeV_40p0mm", 
    # "4Mu_500GeV_0p25GeV_0p004mm", # "4Mu_500GeV_1p2GeV_19p0mm", "4Mu_500GeV_5p0GeV_80p0mm", 
    # "4Mu_500GeV_1p2GeV_0p019mm", "4Mu_500GeV_1p2GeV_0p19mm", "4Mu_500GeV_1p2GeV_1p9mm", "4Mu_500GeV_1p2GeV_9p6mm", 
    "4Mu_500GeV_1p2GeV_19p0mm",]

bkgttj = ["TTJets"]

bkgdyj1 = ["DYJetsToMuMu_M10to50",]
bkgdyj2 = ["DYJetsToMuMu_M50",]

bkgqcd1 = ["QCD_Pt15To20",]
bkgqcd2 = ["QCD_Pt20To30"]
bkgqcd3 = ["QCD_Pt30To50",]
bkgqcd4 = ["QCD_Pt50To80"]
bkgqcd5 = ["QCD_Pt80To120"]
bkgqcd6 = ["QCD_Pt120To170"]
bkgqcd7 = ["QCD_Pt170To300",]
bkgqcd8 = ["QCD_Pt300To470"]
bkgqcd9 = ["QCD_Pt470To600"]
bkgqcd10 = ["QCD_Pt600To800"]
bkgqcd11 = ["QCD_Pt800To1000",] 
bkgqcd12 = ["QCD_Pt1000"]

channels = ["baseNoLj",
            "bkg_base", "bkg_base_2mu2e", "bkg_base_2mu2e_iso", "bkg_base_2mu2e_iso_disp", "bkg_base_2mu2e_iso_disp_dphi",
           ]

In [4]:
runner = processor.Runner(
    # executor=processor.FuturesExecutor(),              # for testing locally
    executor=processor.IterativeExecutor(),              # for testing locally
    # executor=processor.DaskExecutor(client=client),    # for dask
    # schema=NanoAODSchema,
    schema = llpnanoaodschema.LLPNanoAODSchema,
    # maxchunks=1,
    skipbadfiles=True,
   
)

p = sidm_processor.SidmProcessor(
    channels,
    ["BKG_study"],
    unweighted_hist=True, verbose=False
    # verbose=True,
)

In [5]:
# process 2mu2e
fileset2mu = utilities.make_fileset(sig2mu, "llpNanoAOD_v2", max_files = nfiles, location_cfg="signal_2mu2e_v10.yaml")
out2mu = runner.run(fileset2mu, treename="Events", processor_instance=p)
out2mu = out2mu["out"]
coffea.util.save(out2mu, "outputs/bkg_" + vr + "_2mu.coffea")

Output()

Output()

/usr/local/lib/python3.12/site-packages/coffea/nanoevents/schemas/nanoaod.py:264: RuntimeWarning: Missing 
cross-reference index for LowPtElectron_electronIdx => Electron
  warnings.warn(

/usr/local/lib/python3.12/site-packages/coffea/nanoevents/schemas/nanoaod.py:264: RuntimeWarning: Missing 
cross-reference index for LowPtElectron_photonIdx => Photon
  warnings.warn(

/usr/local/lib/python3.12/site-packages/awkward/_nplikes/array_module.py:285: RuntimeWarning: invalid value 
encountered in divide
  return impl(*broadcasted_args, **(kwargs or {}))

...

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

2Mu2E_500GeV_1p2GeV_19p0mm is simulation. Scaling histograms or cutflows according to lumi*xs.
Signal not in xs cfg, assuming 1fb


In [6]:
#process 4mu
fileset4mu = utilities.make_fileset(sig4mu, "llpNanoAOD_v2", max_files = nfiles, location_cfg="signal_4mu_v10.yaml")
out4mu = runner.run(fileset4mu, treename="Events", processor_instance=p)
out4mu = out4mu["out"]
coffea.util.save(out4mu, "outputs/bkg_"+ vr + "_4mu.coffea")

Output()

Output()

/usr/local/lib/python3.12/site-packages/coffea/nanoevents/schemas/nanoaod.py:264: RuntimeWarning: Missing 
cross-reference index for LowPtElectron_electronIdx => Electron
  warnings.warn(

/usr/local/lib/python3.12/site-packages/coffea/nanoevents/schemas/nanoaod.py:264: RuntimeWarning: Missing 
cross-reference index for LowPtElectron_photonIdx => Photon
  warnings.warn(

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

...

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

4Mu_500GeV_1p2GeV_19p0mm is simulation. Scaling histograms or cutflows according to lumi*xs.
Signal not in xs cfg, assuming 1fb


In [7]:
# Process ttj
filesetttj = utilities.make_fileset(bkgttj, "skimmed_llpNanoAOD_v2", max_files = nfiles, location_cfg = "backgrounds.yaml",)
outttj     = runner.run(filesetttj, treename="Events", processor_instance=p)
outttj     = outttj["out"]
coffea.util.save(outttj, "outputs/bkg_" + vr + "_ttj.coffea")

Output()

Output()

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

TTJets is simulation. Scaling histograms or cutflows according to lumi*xs.


In [8]:
# process DYJ
filesetdyj1 = utilities.make_fileset(bkgdyj1, "skimmed_llpNanoAOD_v2", max_files = nfiles, location_cfg = "backgrounds.yaml",)
outdyj1     = runner.run(filesetdyj1, treename="Events", processor_instance=p)
outdyj1     = outdyj1["out"]
coffea.util.save(outdyj1, "outputs/bkg_" + vr + "_dyj1.coffea")

Output()

Output()

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

DYJetsToMuMu_M10to50 is simulation. Scaling histograms or cutflows according to lumi*xs.


In [9]:
# process DYJ
filesetdyj2 = utilities.make_fileset(bkgdyj2, "skimmed_llpNanoAOD_v2", max_files = nfiles, location_cfg = "backgrounds.yaml",)
outdyj2     = runner.run(filesetdyj2, treename="Events", processor_instance=p)
outdyj2     = outdyj2["out"]
coffea.util.save(outdyj2, "outputs/bkg_" + vr + "_dyj2.coffea")

Output()

Output()

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

DYJetsToMuMu_M50 is simulation. Scaling histograms or cutflows according to lumi*xs.


In [10]:
# process qcd1
filesetqcd1    = utilities.make_fileset(bkgqcd1, "skimmed_llpNanoAOD_v2", max_files = nfiles, location_cfg = "backgrounds.yaml",)
outqcd1        = runner.run(filesetqcd1, treename="Events", processor_instance=p)
outqcd1        = outqcd1["out"]
coffea.util.save(outqcd1, "outputs/bkg_" + vr + "_qcd1.coffea")

Output()

Output()

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

QCD_Pt15To20 is simulation. Scaling histograms or cutflows according to lumi*xs.


In [11]:
# process qcd2
filesetqcd2    = utilities.make_fileset(bkgqcd2, "skimmed_llpNanoAOD_v2", max_files = nfiles, location_cfg = "backgrounds.yaml",)
outqcd2        = runner.run(filesetqcd2, treename="Events", processor_instance=p)
outqcd2        = outqcd2["out"]
coffea.util.save(outqcd2, "outputs/bkg_" + vr + "_qcd2.coffea")

Output()

Output()

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

QCD_Pt20To30 is simulation. Scaling histograms or cutflows according to lumi*xs.


In [12]:
# process qcd3
filesetqcd3    = utilities.make_fileset(bkgqcd3, "skimmed_llpNanoAOD_v2", max_files = nfiles, location_cfg = "backgrounds.yaml",)
outqcd3        = runner.run(filesetqcd3, treename="Events", processor_instance=p)
outqcd3        = outqcd3["out"]
coffea.util.save(outqcd3, "outputs/bkg_" + vr + "_qcd3.coffea")

Output()

Output()

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

QCD_Pt30To50 is simulation. Scaling histograms or cutflows according to lumi*xs.


In [13]:
# process qcd4
filesetqcd4    = utilities.make_fileset(bkgqcd4, "skimmed_llpNanoAOD_v2", max_files = nfiles, location_cfg = "backgrounds.yaml",)
outqcd4        = runner.run(filesetqcd4, treename="Events", processor_instance=p)
outqcd4        = outqcd4["out"]
coffea.util.save(outqcd4, "outputs/bkg_" + vr + "_qcd4.coffea")

Output()

Output()

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

QCD_Pt50To80 is simulation. Scaling histograms or cutflows according to lumi*xs.


In [14]:
# process qcd5
filesetqcd5    = utilities.make_fileset(bkgqcd5, "skimmed_llpNanoAOD_v2", max_files = nfiles, location_cfg = "backgrounds.yaml",)
outqcd5        = runner.run(filesetqcd5, treename="Events", processor_instance=p)
outqcd5        = outqcd5["out"]
coffea.util.save(outqcd5, "outputs/bkg_" + vr + "_qcd5.coffea")

Output()

Output()

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

QCD_Pt80To120 is simulation. Scaling histograms or cutflows according to lumi*xs.


In [15]:
# process qcd6
filesetqcd6    = utilities.make_fileset(bkgqcd6, "skimmed_llpNanoAOD_v2", max_files = nfiles, location_cfg = "backgrounds.yaml",)
outqcd6        = runner.run(filesetqcd6, treename="Events", processor_instance=p)
outqcd6        = outqcd6["out"]
coffea.util.save(outqcd6, "outputs/bkg_" + vr + "_qcd6.coffea")

Output()

Output()

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

QCD_Pt120To170 is simulation. Scaling histograms or cutflows according to lumi*xs.


In [16]:
# process qcd7
filesetqcd7    = utilities.make_fileset(bkgqcd7, "skimmed_llpNanoAOD_v2", max_files = nfiles, location_cfg = "backgrounds.yaml",)
outqcd7        = runner.run(filesetqcd7, treename="Events", processor_instance=p)
outqcd7        = outqcd7["out"]
coffea.util.save(outqcd7, "outputs/bkg_" + vr + "_qcd7.coffea")

Output()

Output()

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

QCD_Pt170To300 is simulation. Scaling histograms or cutflows according to lumi*xs.


In [17]:
# process qcd8
filesetqcd8    = utilities.make_fileset(bkgqcd8, "skimmed_llpNanoAOD_v2", max_files = nfiles, location_cfg = "backgrounds.yaml",)
outqcd8        = runner.run(filesetqcd8, treename="Events", processor_instance=p)
outqcd8        = outqcd8["out"]
coffea.util.save(outqcd8, "outputs/bkg_" + vr + "_qcd8.coffea")

Output()

Output()

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

QCD_Pt300To470 is simulation. Scaling histograms or cutflows according to lumi*xs.


In [18]:
# process qcd10
filesetqcd9    = utilities.make_fileset(bkgqcd9, "skimmed_llpNanoAOD_v2", max_files = nfiles, location_cfg = "backgrounds.yaml",)
outqcd9        = runner.run(filesetqcd9, treename="Events", processor_instance=p)
outqcd9        = outqcd9["out"]
coffea.util.save(outqcd9, "outputs/bkg_" + vr + "_qcd9.coffea")

Output()

Output()

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

QCD_Pt470To600 is simulation. Scaling histograms or cutflows according to lumi*xs.


In [19]:
# process qcd2
filesetqcd10    = utilities.make_fileset(bkgqcd10, "skimmed_llpNanoAOD_v2", max_files = nfiles, location_cfg = "backgrounds.yaml",)
outqcd10        = runner.run(filesetqcd10, treename="Events", processor_instance=p)
outqcd10        = outqcd10["out"]
coffea.util.save(outqcd10, "outputs/bkg_" + vr + "_qcd10.coffea")

Output()

Output()

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

QCD_Pt600To800 is simulation. Scaling histograms or cutflows according to lumi*xs.


In [20]:
# process qcd2
filesetqcd11    = utilities.make_fileset(bkgqcd11, "skimmed_llpNanoAOD_v2", max_files = nfiles, location_cfg = "backgrounds.yaml",)
outqcd11        = runner.run(filesetqcd11, treename="Events", processor_instance=p)
outqcd11        = outqcd11["out"]
coffea.util.save(outqcd11, "outputs/bkg_" + vr + "_qcd11.coffea")

Output()

Output()

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

QCD_Pt800To1000 is simulation. Scaling histograms or cutflows according to lumi*xs.


In [21]:
# process qcd2
filesetqcd12    = utilities.make_fileset(bkgqcd12, "skimmed_llpNanoAOD_v2", max_files = nfiles, location_cfg = "backgrounds.yaml",)
outqcd12        = runner.run(filesetqcd12, treename="Events", processor_instance=p)
outqcd12        = outqcd12["out"]
coffea.util.save(outqcd12, "outputs/bkg_" + vr + "_qcd12.coffea")

Output()

Output()

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

Warning: Unable to apply all for nested dsaMuons collection. Skipping.... no field named 'good_matched_muons'

QCD_Pt1000 is simulation. Scaling histograms or cutflows according to lumi*xs.


In [22]:
objs

NameError: name 'objs' is not defined

In [ ]:
# out_all = out_2mu | out_4mu | out_dyj | out_ttj | out_qcd | out_ocd
# outall = {
#     **out2mu, 
#     **out4mu,
#     **outdyj,
#     **outttj,
#     **outqcd, 
#     **outocd,
# }
# coffea.util.save(out_all, f"outputs/bkg_{vr}.coffea")

In [ ]:
raise SystemExit("Notebook stopped intentionally after this cell.")

# Start plottong

In [ ]:
output = coffea.util.load("outputs/bkg_"+ vr +".coffea")

lab_mx = ["200", "500", "800", "1000"]
lab_xy = ["0.3", "3.0", "30", "150", "300"]
lab_zd = ["0.25", "1.2", "5.0"]

In [ ]:
# Electron Variables
htplot = ["genE_pt",
          "electron_pt",
          # "muon_dxy",
          # "electron_pt",
          # "dsaMuon_pt"
          # "dsaMuon_dxy"
          # "muon_muon_dR",
         ]

htname = [r"Gen E $p_T$",
          r"Ele $p_T$",
          # r"Muon $d_0$",
          # r"ele $p_T$",
          # r"dsamu $p_T$",
          # r"dsamu $d_0$",
          # r"Muon $\Delta R$",
         ]

# changing Bs
for ik, ht in enumerate(htplot):
    plt.subplots(1, 1, figsize=(16, 10))
    for ij, sss in enumerate(mxx2):
        utilities.plot(output[sss]["hists"][ht][ch1, :], label = lab_mx[ij])
    plt.title(htname[ik])
    plt.ylabel("Events")
    plt.yscale("log")
    plt.legend(title="mxx [GeV]", alignment="left", loc=0)
    plt.savefig(f"clean_plots/ch5_bkg_{vr}_mxx_{ht}_.pdf", bbox_inches="tight", dpi=300)

# changing zd
# for ik, ht in enumerate(htplot):
#     plt.subplots(1, 1, figsize=(16, 10))
#     for ij, sss in enumerate(mzd4):
#         utilities.plot(output[sss]["hists"][ht][ch1, :], label = lab_zd[ij])
#     plt.title(htname[ik])
#     plt.ylabel("Events")
#     plt.yscale("log")
#     plt.legend(title=r"$z_D$ [GeV]", alignment="left", loc=0)
#     plt.savefig(f"clean_plots/ch5_bkg_{vr}_mzd_{ht}_.pdf", bbox_inches="tight", dpi=300)

# changing lxy
# for ik, ht in enumerate(htplot):
#     plt.subplots(1, 1, figsize=(16, 10))
#     for ij, sss in enumerate(lxy4):
#         utilities.plot(output[sss]["hists"][ht][ch1, :], label = lab_xy[ij])
#     plt.title(htname[ik])
#     plt.ylabel("Events")
#     plt.yscale("log")
#     plt.legend(title=r"$L_{xy}$ [cm]", alignment="left", loc=0)
#     plt.savefig(f"clean_plots/ch5_bkg_{vr}_lxy_{ht}_.pdf", bbox_inches="tight", dpi=300)

In [ ]:
# Muon Variables
htplot = ["genMu_pt",
          "muon_pt",
          "muon_dxy",
          # "electron_pt",
          "dsaMuon_pt",
          "dsaMuon_dxy",
          # "muon_muon_dR",
         ]

htname = [r"Gen Muon $p_T$",
          r"Muon $p_T$",
          r"Muon $d_0$",
          # r"ele $p_T$",
          r"dsamu $p_T$",
          r"dsamu $d_0$",
          # r"Muon $\Delta R$",
         ]

# changing Bs
for ik, ht in enumerate(htplot):
    plt.subplots(1, 1, figsize=(16, 10))
    for ij, sss in enumerate(mxx4):
        utilities.plot(output[sss]["hists"][ht][ch1, :], label = lab_mx[ij])
    plt.title(htname[ik])
    plt.ylabel("Events")
    plt.yscale("log")
    plt.legend(title="mxx [GeV]", alignment="left", loc=0)
    # plt.savefig(f"clean_plots/ch5_bkg_{vr}_mxx_{ht}_.pdf", bbox_inches="tight", dpi=300)

# changing zd
# for ik, ht in enumerate(htplot):
#     plt.subplots(1, 1, figsize=(16, 10))
#     for ij, sss in enumerate(mzd4):
#         utilities.plot(output[sss]["hists"][ht][ch1, :], label = lab_zd[ij])
#     plt.title(htname[ik])
#     plt.ylabel("Events")
#     plt.yscale("log")
#     plt.legend(title=r"$z_D$ [GeV]", alignment="left", loc=0)
#     plt.savefig(f"clean_plots/ch5_bkg_{vr}_mzd_{ht}_.pdf", bbox_inches="tight", dpi=300)

# changing lxy
# for ik, ht in enumerate(htplot):
#     plt.subplots(1, 1, figsize=(16, 10))
#     for ij, sss in enumerate(lxy4):
#         utilities.plot(output[sss]["hists"][ht][ch1, :], label = lab_xy[ij])
#     plt.title(htname[ik])
#     plt.ylabel("Events")
#     plt.yscale("log")
#     plt.legend(title=r"$L_{xy}$ [cm]", alignment="left", loc=0)
#     plt.savefig(f"clean_plots/ch5_bkg_{vr}_lxy_{ht}_.pdf", bbox_inches="tight", dpi=300)

In [ ]:
# LJ Variables signal only

htplot = ["lj_pt",
          "lj_iso",
          "lj_muon_pt",
          "lj0_pt",
          "lj1_pt",
 "lj_e",
 "lj0_e",
 "lj1_e",
 "lj0_dRSpread",
 "lj1_dRSpread",
 "lj_electronN",
 "lj_photonN",
 "lj_electronPhotonN",
 "lj_muonN",
 "lj_dsaMuN",
 "lj_pfMuN",
 "lj_muon_pt",
 "lj_pfMuon_pt",
"lj_dsaMuon_pt",
 "lj_electron_pt",
 "lj_photon_pt",
 "lj_muon_dxy",
 "lj_pfMuon_dxy",
 "lj_dsaMuon_dxy",
 "lj_dsaMuon_dz",
 "lj_electron_dxy",
         ]

htname = [r"LJs $p_T$",
          "lj_iso",
          "lj_muon_pt",
          "lj0_pt",
          "lj1_pt",
 "lj_e",
 "lj0_e",
 "lj1_e",
 "lj0_dRSpread",
 "lj1_dRSpread",
 "lj_electronN",
 "lj_photonN",
 "lj_electronPhotonN",
 "lj_muonN",
 "lj_dsaMuN",
 "lj_pfMuN",
 "lj_muon_pt",
 "lj_pfMuon_pt",
"lj_dsaMuon_pt",
 "lj_electron_pt",
 "lj_photon_pt",
 "lj_muon_dxy",
 "lj_pfMuon_dxy",
 "lj_dsaMuon_dxy",
 "lj_dsaMuon_dz",
 "lj_electron_dxy",
          # r"LJs $\Delta \Phi$",
          # r"LJs $\Delta R$",
          # r"LJs $\eta$",
         ]

ch = ch5

# changing Bs 4mu
for ik, ht in enumerate(htplot):
    plt.subplots(1, 2, figsize=(32, 10))

    plt.subplot(1, 2, 1)
    for ij, sss in enumerate(mxx2):
        utilities.plot(output[sss]["hists"][ht][ch7, :], label = lab_mx[ij])
    plt.title(htname[ik]+": 2mu")
    plt.ylabel("Events")
    plt.yscale("log")
    plt.legend(title="mxx [GeV]", alignment="left", loc=0)

    plt.subplot(1, 2, 2)
    for ij, sss in enumerate(mxx4):
        utilities.plot(output[sss]["hists"][ht][ch8, :], label = lab_mx[ij])

    plt.title(htname[ik]+": 4mu")
    plt.ylabel("Events")
    plt.yscale("log")
    plt.legend(title="mxx [GeV]", alignment="left", loc=0)
    # plt.savefig(f"clean_plots/ch5_{vr}_mxx_{ht}_.pdf", bbox_inches="tight", dpi=300)

In [ ]:
# LJ-LJ Variables signal only

htplot = ["lj_lj_invmass",
          # "lj_lj_absdphi",
          # "lj_lj_absdR",
          # "lj_lj_absdeta"
         ]

htname = [r"LJs inv Mass",
          # r"LJs $\Delta \Phi$",
          # r"LJs $\Delta R$",
          # r"LJs $\eta$",
         ]

ch = ch5

# changing Bs 4mu
for ik, ht in enumerate(htplot):
    plt.subplots(1, 2, figsize=(32, 10))

    plt.subplot(1, 2, 1)
    for ij, sss in enumerate(mxx2):
        utilities.plot(output[sss]["hists"][ht][ch7, :], label = lab_mx[ij])
    plt.title(htname[ik]+": 2mu")
    plt.ylabel("Events")
    plt.yscale("log")
    plt.legend(title="mxx [GeV]", alignment="left", loc=0)

    plt.subplot(1, 2, 2)
    for ij, sss in enumerate(mxx4):
        utilities.plot(output[sss]["hists"][ht][ch8, :], label = lab_mx[ij])

    plt.title(htname[ik]+": 4mu")
    plt.ylabel("Events")
    plt.yscale("log")
    plt.legend(title="mxx [GeV]", alignment="left", loc=0)
    # plt.savefig(f"clean_plots/ch5_{vr}_mxx_{ht}_.pdf", bbox_inches="tight", dpi=300)


# for ij, sss in enumerate(mxx4):
#     for ik, ht in enumerate(htplot):
#         plt.subplots(1, 1, figsize=(16, 10))
#         for ch in cha:
#             utilities.plot(output[sss]["hists"][ht][ch, :], label = sss+ht+ch)#lab_mx[ij])


In [ ]:
# LJ-LJ Variables signal only different channels

htplot = ["lj_lj_invmass",
          # "lj_lj_absdphi",
          # "lj_lj_absdR",
          # "lj_lj_absdeta"
         ]

htname = [r"LJs inv Mass",
          # r"LJs $\Delta \Phi$",
          # r"LJs $\Delta R$",
          # r"LJs $\eta$",
         ]

ch = ch5

# # changing Bs 4mu
# for ik, ht in enumerate(htplot):
#     plt.subplots(1, 2, figsize=(32, 10))

#     plt.subplot(1, 2, 1)
#     for ij, sss in enumerate(mxx2):
#         utilities.plot(output[sss]["hists"][ht][ch, :], label = lab_mx[ij])
#     plt.title(htname[ik]+": 2mu")
#     plt.ylabel("Events")
#     plt.yscale("log")
#     plt.legend(title="mxx [GeV]", alignment="left", loc=0)

#     plt.subplot(1, 2, 2)
#     for ij, sss in enumerate(mxx4):
#         utilities.plot(output[sss]["hists"][ht][ch6, :], label = lab_mx[ij])

#     plt.title(htname[ik]+": 4mu")
#     plt.ylabel("Events")
#     plt.yscale("log")
#     plt.legend(title="mxx [GeV]", alignment="left", loc=0)
#     # plt.savefig(f"clean_plots/ch5_{vr}_mxx_{ht}_.pdf", bbox_inches="tight", dpi=300)


for ij, sss in enumerate(mxx4):
    for ik, ht in enumerate(htplot):
        plt.subplots(1, 1, figsize=(16, 10))
        for ch in cha:
            utilities.plot(output[sss]["hists"][ht][ch, :], label = ch)#lab_mx[ij])
        plt.legend(title="mxx [GeV]", alignment="left", loc=0)


In [ ]:
# LJ-LJ Variables
htplot = ["lj_lj_invmass",
          # "lj_lj_absdphi",
          # "lj_lj_absdR",
          # "lj_lj_absdeta"
         ]

htname = [r"LJs inv Mass",
          # r"LJs $\Delta \Phi$",
          # r"LJs $\Delta R$",
          # r"LJs $\eta$",
         ]


# changing Bs 4mu
for ik, ht in enumerate(htplot):
    plt.subplots(1, 1, figsize=(16, 10))
    for ij, sss in enumerate(mxx4):
        utilities.plot(output[sss]["hists"][ht][ch6, :], label = lab_mx[ij])
        signal = output[sss]["hists"][ht][ch6, ::2j]

    TTJ = 0
    for ij, ttj in enumerate(bkgttj):
        TTJ = TTJ + output[ttj]["hists"][ht][ch6, ::2j]
    utilities.plot(TTJ, label = "TTJets")

    
    QCD = 0
    for qcd in bkgqcd:
        QCD = QCD + output[qcd]["hists"][ht][ch6, ::2j]
    utilities.plot(QCD, label = "QCD", density=False, color="green")

    DYJ = 0 
    for dyj in bkgdyj:
        DYJ = DYJ + output[dyj]["hists"][ht][ch6, ::2j]
    utilities.plot(DYJ, label = "DY", density=False, color="blue")
    # print(type(DYJ))
    # print(inspect.signature(utilities.plot))
    
    plt.title(htname[ik]+": 4mu")
    plt.ylabel("Events")
    plt.yscale("log")
    plt.legend(title="mxx [GeV]", alignment="left", loc=0)
    # plt.savefig(f"clean_plots/ch5_{vr}_mxx_{ht}_.pdf", bbox_inches="tight", dpi=300)


# changing Bs 2mu
# for ik, ht in enumerate(htplot):
#     plt.subplots(1, 1, figsize=(16, 10))
#     for ij, sss in enumerate(mxx2):
#         utilities.plot(output[sss]["hists"][ht][chan, :], label = lab_mx[ij])
#     for ij, sss in enumerate(bkgttj):
#         utilities.plot(output[sss]["hists"][ht][chan, :], label = sss)
#     for ij, sss in enumerate(bkgdyj):
#         utilities.plot(output[sss]["hists"][ht][chan, :], label = sss)
#     plt.title(htname[ik]+r": 2mu2e")
#     plt.ylabel("Events")
#     plt.yscale("log")
#     plt.legend(title="mxx [GeV]", alignment="left", loc=0)
    # plt.savefig(f"clean_plots/ch5_{vr}_mxx_{ht}_.pdf", bbox_inches="tight", dpi=300)

    
# # changing zd
# for ik, ht in enumerate(htplot):
#     plt.subplots(1, 1, figsize=(16, 10))
#     for ij, sss in enumerate(mzd4):
#         utilities.plot(output[sss]["hists"][ht][ch1, :], label = lab_zd[ij])
#     plt.title(htname[ik])
#     plt.ylabel("Events")
#     plt.yscale("log")
#     plt.legend(title=r"$z_D$ [GeV]", alignment="left", loc=0)
#     plt.savefig(f"clean_plots/ch5_{vr}_mzd_{ht}_.pdf", bbox_inches="tight", dpi=300)

# # changing lxy
# for ik, ht in enumerate(htplot):
#     plt.subplots(1, 1, figsize=(16, 10))
#     for ij, sss in enumerate(lxy4):
#         utilities.plot(output[sss]["hists"][ht][ch1, :], label = lab_xy[ij])
#     plt.title(htname[ik])
#     plt.ylabel("Events")
#     plt.yscale("log")
#     plt.legend(title=r"$L_{xy}$ [cm]", alignment="left", loc=0)
#     plt.savefig(f"clean_plots/ch5_{vr}_lxy_{ht}_.pdf", bbox_inches="tight", dpi=300)

In [ ]:
import matplotlib.pyplot as plt
import mplhep as hep

hep.histplot(
    [DYJ, TTJ, QCD],
    stack=True,
    histtype="fill",
    color=["gold", "red", "deepskyblue"],
    label=["DY", "TTJets", "QCD"],
)
plt.legend()
# plt.show()


# Signal (overlay)
hep.histplot(
    signal,
    histtype="step",
    linewidth=3,
    color="black",
    label="SIDM Signal",
    ax=ax,
)

ax.legend()
ax.set_xlabel(r"$m_{\mu\mu}$ [GeV]")
ax.set_ylabel("Events")
plt.show()



In [ ]:
import matplotlib.pyplot as plt
import mplhep as hep

fig, ax = plt.subplots()

# Backgrounds (stacked)
hep.histplot(
    [DYJ, TTJ, QCD],
    stack=True,
    histtype="fill",
    color=["gold", "red", "deepskyblue"],
    label=["DY", "TTJets", "QCD"],
    ax=ax,
)

# Signal (overlay)
hep.histplot(
    signal,
    histtype="step",
    linewidth=3,
    color="black",
    label="SIDM Signal",
    ax=ax,
)

ax.legend()
ax.set_xlabel(r"$m_{\mu\mu}$ [GeV]")
ax.set_ylabel("Events")
plt.show()

In [ ]:
# Muon Variables
htplot = ["muon_pt",
          "muon_dxy",
          "electron_pt",
          "dsaMuon_pt"
          "dsaMuon_dxy"
          # "muon_muon_dR",
         ]

htname = [r"Muon $p_T$",
          r"Muon $d_0$",
          r"ele $p_T$",
          r"dsamu $p_T$",
          r"dsamu $d_0$",
          # r"Muon $\Delta R$",
         ]

# changing Bs
for ik, ht in enumerate(htplot):
    plt.subplots(1, 1, figsize=(16, 10))
    for ij, sss in enumerate(mxx4):
        utilities.plot(output[sss]["hists"][ht][ch1, :], label = lab_mx[ij])
    plt.title(htname[ik])
    plt.ylabel("Events")
    plt.yscale("log")
    plt.legend(title="mxx [GeV]", alignment="left", loc=0)
    plt.savefig(f"clean_plots/ch5_bkg_{vr}_mxx_{ht}_.pdf", bbox_inches="tight", dpi=300)

# changing zd
# for ik, ht in enumerate(htplot):
#     plt.subplots(1, 1, figsize=(16, 10))
#     for ij, sss in enumerate(mzd4):
#         utilities.plot(output[sss]["hists"][ht][ch1, :], label = lab_zd[ij])
#     plt.title(htname[ik])
#     plt.ylabel("Events")
#     plt.yscale("log")
#     plt.legend(title=r"$z_D$ [GeV]", alignment="left", loc=0)
#     plt.savefig(f"clean_plots/ch5_bkg_{vr}_mzd_{ht}_.pdf", bbox_inches="tight", dpi=300)

# changing lxy
# for ik, ht in enumerate(htplot):
#     plt.subplots(1, 1, figsize=(16, 10))
#     for ij, sss in enumerate(lxy4):
#         utilities.plot(output[sss]["hists"][ht][ch1, :], label = lab_xy[ij])
#     plt.title(htname[ik])
#     plt.ylabel("Events")
#     plt.yscale("log")
#     plt.legend(title=r"$L_{xy}$ [cm]", alignment="left", loc=0)
#     plt.savefig(f"clean_plots/ch5_bkg_{vr}_lxy_{ht}_.pdf", bbox_inches="tight", dpi=300)

In [ ]:
raise SystemExit("Notebook stopped intentionally after this cell.")

In [ ]:
# Dark Photon Variables

htplot = ["genAs_pt", "genA_n", "genAs_eta", "genAs_phi", "genAs_mass"]
htname = [r"Dark Photon $p_T$", r"Number of $z_D$", r"Dark Photon $\eta$", r"Dark Photon $\phi$", r"Dark Photon Mass"]

#changing Bs mass
for ik, ht in enumerate(htplot):
    plt.subplots(1, 1, figsize=(16, 10))
    for ij, sss in enumerate(mxx2):
        utilities.plot(output[sss]["hists"][ht][ch1, :], label = lab_mx[ij])
    plt.title(htname[ik])
    plt.ylabel("Events")
    plt.yscale("log")
    plt.legend(title="mxx [GeV]", alignment="left", loc=0)
    plt.savefig(f"clean_plots/ch5_{vr}_mxx_{ht}_.pdf", bbox_inches="tight", dpi=300)

#changing lxy
for ik, ht in enumerate(htplot):
    plt.subplots(1, 1, figsize=(16, 10))
    for ij, sss in enumerate(lxy2):
        utilities.plot(output[sss]["hists"][ht][ch1, :], label = lab_xy[ij])
    plt.title(htname[ik])
    plt.ylabel("Events")
    plt.yscale("log")
    plt.legend(title=r"$L_{xy}$ [cm]", alignment="left", loc=0)
    plt.savefig(f"clean_plots/ch5_{vr}_lxy_{ht}_.pdf", bbox_inches="tight", dpi=300)

In [ ]:
# LJ-LJ Variables
htplot = ["lj_lj_absdphi",
          "lj_lj_invmass",
          "lj_lj_absdR",
          "lj_lj_absdeta"
         ]

htname = [r"LJs $\Delta \Phi$",
          r"LJs inv Mass$",
          r"LJs $\Delta R$",
          r"LJs $\eta$",
         ]

# changing Bs
for ik, ht in enumerate(htplot):
    plt.subplots(1, 1, figsize=(16, 10))
    for ij, sss in enumerate(mxx4):
        utilities.plot(output[sss]["hists"][ht][ch1, :], label = lab_mx[ij])
    plt.title(htname[ik])
    plt.ylabel("Events")
    plt.yscale("log")
    plt.legend(title="mxx [GeV]", alignment="left", loc=0)
    plt.savefig(f"clean_plots/ch5_{vr}_mxx_{ht}_.pdf", bbox_inches="tight", dpi=300)

# changing zd
for ik, ht in enumerate(htplot):
    plt.subplots(1, 1, figsize=(16, 10))
    for ij, sss in enumerate(mzd4):
        utilities.plot(output[sss]["hists"][ht][ch1, :], label = lab_zd[ij])
    plt.title(htname[ik])
    plt.ylabel("Events")
    plt.yscale("log")
    plt.legend(title=r"$z_D$ [GeV]", alignment="left", loc=0)
    plt.savefig(f"clean_plots/ch5_{vr}_mzd_{ht}_.pdf", bbox_inches="tight", dpi=300)

# changing lxy
for ik, ht in enumerate(htplot):
    plt.subplots(1, 1, figsize=(16, 10))
    for ij, sss in enumerate(lxy4):
        utilities.plot(output[sss]["hists"][ht][ch1, :], label = lab_xy[ij])
    plt.title(htname[ik])
    plt.ylabel("Events")
    plt.yscale("log")
    plt.legend(title=r"$L_{xy}$ [cm]", alignment="left", loc=0)
    plt.savefig(f"clean_plots/ch5_{vr}_lxy_{ht}_.pdf", bbox_inches="tight", dpi=300)

In [ ]:
raise SystemExit("Notebook stopped intentionally after this cell.")